# Word2Vec

**学習目標**

1. Word2Vec モデルの構築方法を学ぶ
2. Word2Vec 用の訓練データを準備する
3. Word2Vec モデルを訓練する（このラボではスキップグラムモデルを構築します）
4. Embedding Projector を使用して埋め込みを可視化および分析する方法を学ぶ




## はじめに
Word2Vec は単一のアルゴリズムではなく、大規模なデータセットから単語の埋め込みを学習するために使用できるモデルアーキテクチャと最適化手法のファミリです。Word2Vec を通じて学習された埋め込みは、さまざまなダウンストリームの自然言語処理タスクで成功を収めています。

注: このノートブックは [Efficient Estimation of Word Representations in Vector Space](https://arxiv.org/pdf/1301.3781.pdf) および [Distributed Representations of Words and Phrases and their Compositionality](https://papers.nips.cc/paper/5021-distributed-representations-of-words-and-phrases-and-their-compositionality.pdf) に基づいています。論文の完全な再実装ではなく、主要なアイデアを説明することを目的としています。

これらの論文では、単語の表現を学習するための 2 つの手法が提案されています。

*   **Continuous Bag-of-Words（CBOW）モデル**: 周囲のコンテキスト単語に基づいて中央の単語を予測します。コンテキストは、現在の中央単語の前後のいくつかの単語で構成されます。コンテキスト内の単語の順序は重要ではないため、このアーキテクチャは Bag-of-Words モデルと呼ばれます。
*   **Continuous Skip-gram モデル**: 同じ文内の現在の単語の前後の特定の範囲内にある単語を予測します。


このノートブックではスキップグラムのアプローチを使用します。まず、単一の文を使用してスキップグラムやその他の概念を説明します。次に、小さなデータセットで独自の Word2Vec モデルを訓練します。このノートブックには、学習した埋め込みをエクスポートし、[TensorFlow Embedding Projector](http://projector.tensorflow.org/) で可視化するためのコードも含まれています。

## スキップグラムとネガティブサンプリング

Bag-of-Words モデルが周囲のコンテキストが与えられたときに単語を予測するのに対し、スキップグラムモデルは単語自体が与えられたときにその単語のコンテキスト（または周囲の単語）を予測します。モデルはスキップグラムで訓練されます。スキップグラムとは、トークンをスキップできる n-gram です（例については下の図を参照）。単語のコンテキストは、`context_word` が `target_word` の周囲のコンテキストに現れる `(target_word, context_word)` のスキップグラムペアのセットとして表現できます。

8 単語からなる次の文を考えてみましょう。
> The wide road shimmered in the hot sun.

この文の 8 単語のそれぞれのコンテキスト単語は、ウィンドウサイズによって定義されます。ウィンドウサイズは、`context word`（コンテキスト単語）と見なすことができる `target_word`（ターゲット単語）の両側の単語の範囲を決定します。異なるウィンドウサイズに基づくターゲット単語のスキップグラムの表を見てみましょう。

注: このラボでは、ウィンドウサイズ *n* は各側に n 単語を意味し、単語を挟んで合計 2*n+1 単語のウィンドウ幅になります。

![word2vec_skipgrams](assets/word2vec_skipgram.png)

スキップグラムモデルの訓練目標は、ターゲット単語が与えられたときにコンテキスト単語を予測する確率を最大化することです。単語のシーケンス *w<sub>1</sub>, w<sub>2</sub>, ... w<sub>T</sub>* に対する目的関数は、平均対数確率として次のように記述できます。

![word2vec_skipgram_objective](assets/word2vec_skipgram_objective.png)

ここで `c` は訓練コンテキストのサイズです。基本的なスキップグラムの定式化では、softmax 関数を使用してこの確率を定義します。

![word2vec_full_softmax](assets/word2vec_full_softmax.png)

ここで *v* と *v<sup>'<sup>* は単語のターゲットベクトル表現とコンテキストベクトル表現であり、*W* は語彙サイズです。ここで *v<sub>0* と *v<sub>1* は勾配降下法によって更新されるモデルパラメータです。

この定式化の分母を計算するには、語彙全体（多くの場合 10<sup>5</sup>〜10<sup>7</sup> 語と大規模）にわたる完全な softmax を実行する必要があります。

[Noise Contrastive Estimation（NCE）](https://www.tensorflow.org/api_docs/python/tf/nn/nce_loss) 損失関数は、完全な softmax の効率的な近似です。単語の分布をモデル化するのではなく単語の埋め込みを学習することを目的とする場合、NCE 損失はネガティブサンプリングを使用するように[簡略化](https://papers.nips.cc/paper/5021-distributed-representations-of-words-and-phrases-and-their-compositionality.pdf)できます。

ターゲット単語に対する簡略化されたネガティブサンプリングの目的は、コンテキスト単語を、単語のノイズ分布 *P<sub>n</sub>(w)* から抽出された *num_ns* 個のネガティブサンプルと区別することです。より正確には、語彙全体の完全な softmax の効率的な近似は、スキップグラムペアに対して、ターゲット単語の損失をコンテキスト単語と *num_ns* 個のネガティブサンプルとの間の分類問題として設定することです。

ネガティブサンプルは、context_word が target_word の `window_size` の近傍に現れないような (target_word, context_word) ペアとして定義されます。例文の場合、これらは（`window_size` が 2 の場合の）潜在的なネガティブサンプルのいくつかです。

```
(hot, shimmered)
(wide, hot)
(wide, sun)
```

次のセクションでは、単一の文に対してスキップグラムとネガティブサンプルを生成します。また、サブサンプリング手法について学び、チュートリアルの後半で正と負の訓練サンプルに対する分類モデルを訓練します。

## セットアップ

In [ ]:
import os
import warnings

warnings.filterwarnings("ignore")
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

# バックエンドを選択: "tensorflow", "jax", または "torch"
os.environ["KERAS_BACKEND"] = "tensorflow"
import io
import re
import string

import keras
import numpy as np
import tensorflow as tf  # tf.data パイプライン用に保持
import tqdm
from keras import layers, ops  # ops にはバックエンドに依存しない数学関数が含まれます

下のセルを使用して、TensorFlow と Keras のバージョンを確認してください。

In [ ]:
print(f"TensorFlow version: {tf.version.VERSION}")
print(f"Keras version: {keras.__version__}")

In [ ]:
# 必要に応じて以下を変更してください
PROJECT = !gcloud config get-value project  # noqa: E999
PROJECT = PROJECT[0]
BUCKET = PROJECT
REGION = "us-central1"

OUTDIR = f"gs://{BUCKET}/text_models"

%env PROJECT=$PROJECT
%env BUCKET=$BUCKET
%env REGION=$REGION
%env OUTDIR=$OUTDIR

In [ ]:
SEED = 42
AUTOTUNE = tf.data.AUTOTUNE

### 例文のベクトル化

次の文を考えてみましょう。

> The wide road shimmered in the hot sun.

文をトークン化します。

In [ ]:
sentence = "The wide road shimmered in the hot sun"
tokens = list(sentence.lower().split())
print(len(tokens))

トークンから整数インデックスへのマッピングを保存する語彙（vocabulary）を作成します。

In [ ]:
vocab, index = {}, 1  # インデックスは 1 から開始
vocab["<pad>"] = 0  # パディングトークンを追加
for token in tokens:
    if token not in vocab:
        vocab[token] = index
        index += 1
vocab_size = len(vocab)
print(vocab)

整数インデックスからトークンへのマッピングを保存する逆語彙（inverse vocabulary）を作成します。

In [ ]:
inverse_vocab = {index: token for token, index in vocab.items()}
print(inverse_vocab)

文をベクトル化します。

In [ ]:
example_sequence = [vocab[word] for word in tokens]
print(example_sequence)

### 1 つの文からのスキップグラムの生成

`tf.keras.preprocessing.sequence` モジュールは、Word2Vec のデータ準備を簡素化する便利な関数を提供します。`tf.keras.preprocessing.sequence.skipgrams` を使用して、指定された `window_size` で `[0, vocab_size)` の範囲のトークンから `example_sequence` のスキップグラムペアを生成できます。

注: この関数によって生成されたネガティブサンプルをバッチ処理するには少しコードが必要なため、ここでは `negative_samples` を `0` に設定しています。次のセクションで、別の関数を使用してネガティブサンプリングを実行します。

In [ ]:
window_size = 2
positive_skip_grams, _ = tf.keras.preprocessing.sequence.skipgrams(
    example_sequence,
    vocabulary_size=vocab_size,
    window_size=window_size,
    negative_samples=0,
    seed=SEED
)
print(len(positive_skip_grams))

いくつかの正のスキップグラムを見てみましょう。

In [ ]:
for target, context in positive_skip_grams[:5]:
    print(
        f"({target}, {context}): ({inverse_vocab[target]}, {inverse_vocab[context]})"
    )

### 1 つのスキップグラムに対するネガティブサンプリング

`skipgrams` 関数は、指定されたウィンドウ幅をスライドさせることによって、すべての正のスキップグラムペアを返します。訓練用のネガティブサンプルとして機能する追加のスキップグラムペアを生成するには、語彙からランダムな単語をサンプリングする必要があります。`tf.random.log_uniform_candidate_sampler` 関数を使用して、ウィンドウ内の特定のターゲット単語に対して `num_ns` 個のネガティブサンプルをサンプリングします。1 つのスキップグラムのターゲット単語に対してこの関数を呼び出し、コンテキスト単語を正解クラスとして渡すことで、サンプリング対象から除外できます。

重要ポイント: 正のコンテキスト単語あたりのネガティブサンプル数 *num_ns* は、小さなデータセットでは [5, 20] の間が最も効果的であることが[示されています](https://papers.nips.cc/paper/5021-distributed-representations-of-words-and-phrases-and-their-compositionality.pdf)。一方、大きなデータセットでは [2, 5] の間の *num_ns* で十分です。

In [ ]:
# 1 つの正のスキップグラムのターゲット単語とコンテキスト単語を取得
target_word, context_word = positive_skip_grams[0]

# 正のコンテキストあたりのネガティブサンプル数を設定
num_ns = 4

context_class = ops.reshape(
    ops.convert_to_tensor(context_word, dtype="int64"), (1, 1)
)
negative_sampling_candidates, _, _ = tf.random.log_uniform_candidate_sampler(
    true_classes=context_class,  # 「正（positive）」としてサンプリングされるべきクラス
    num_true=1,  # 各正のスキップグラムは 1 つの正のコンテキストクラスを持つ
    num_sampled=num_ns,  # サンプリングする負のコンテキスト単語の数
    unique=True,  # すべてのネガティブサンプルは一意である必要がある
    range_max=vocab_size,  # サンプルのインデックスを [0, vocab_size] から選択
    seed=SEED,  # 再現性のためのシード値
    name="negative_sampling",  # この操作の名前
)
print(negative_sampling_candidates)
print([inverse_vocab[index.numpy()] for index in negative_sampling_candidates])

### 1 つの訓練サンプルの構築

特定の正の `(target_word, context_word)` スキップグラムに対して、`target_word` のウィンドウサイズの近傍に現れない `num_ns` 個のネガティブサンプリングされたコンテキスト単語も得られました。`1` 個の正の `context_word` と `num_ns` 個の負のコンテキスト単語を 1 つのテンソルにバッチ化します。これにより、各ターゲット単語に対して、正のスキップグラム（ラベル `1`）とネガティブサンプル（ラベル `0`）のセットが生成されます。

In [ ]:
# （次のステップで）連結を使用できるように次元を削減
squeezed_context_class = ops.squeeze(context_class, 1)

# 正のコンテキスト単語とネガティブサンプリングされた単語を連結
context = ops.concatenate(
    [squeezed_context_class, negative_sampling_candidates], 0
)

# 最初のコンテキスト単語を `1`（正）とし、その後に `num_ns` 個の `0`（負）をラベル付け
label = ops.convert_to_tensor([1] + [0] * num_ns, dtype="int64")
target = target_word

上記のスキップグラムの例から、ターゲット単語に対するコンテキストと対応するラベルを見てみましょう。

In [ ]:
print(f"target_index    : {target}")
print(f"target_word     : {inverse_vocab[target_word]}")
print(f"context_indices : {context}")
print(f"context_words   : {[inverse_vocab[c.numpy()] for c in context]}")
print(f"label           : {label}")

`(target, context, label)` テンソルのタプルは、スキップグラムネガティブサンプリング Word2Vec モデルを訓練するための 1 つの訓練サンプルを構成します。target の形状が `(1,)` であるのに対し、context と label の形状は `(1+num_ns,)` であることに注目してください。

In [ ]:
print(f"target  : {target}")
print(f"context : {context}")
print(f"label   : {label}")

### まとめ

この図は、文から訓練サンプルを生成する手順をまとめたものです。

![word2vec_negative_sampling](assets/word2vec_negative_sampling.png)

## ラボ課題 1

### スキップグラムサンプリングテーブル

大規模なデータセットは、ストップワードなどのより頻度の高い単語が多く含まれる、より大きな語彙を意味します。頻繁に出現する単語（`the`、`is`、`on` など）のサンプリングから得られた訓練サンプルは、モデルが学習するための有用な情報をあまり付加しません。[Mikolov 氏ら](https://papers.nips.cc/paper/5021-distributed-representations-of-words-and-phrases-and-their-compositionality.pdf) は、埋め込みの品質を向上させるための有益な実践として、頻出単語のサブサンプリングを提案しています。

`tf.keras.preprocessing.sequence.skipgrams` 関数は、任意のトークンをサンプリングする確率をエンコードするサンプリングテーブル引数を受け入れます。`tf.keras.preprocessing.sequence.make_sampling_table` を使用して、単語頻度のランクに基づく確率的サンプリングテーブルを生成し、それを `skipgrams` 関数に渡すことができます。`vocab_size` が 10 の場合のサンプリング確率を見てみましょう。

In [ ]:
sampling_table = tf.keras.preprocessing.sequence.make_sampling_table(size=10)
print(sampling_table)

`sampling_table[i]` は、データセット内で i 番目に一般的な単語をサンプリングする確率を示します。この関数は、サンプリングのための単語頻度が[ジップの法則（Zipf's distribution）](https://en.wikipedia.org/wiki/Zipf%27s_law)に従うことを前提としています。

重要ポイント: `tf.random.log_uniform_candidate_sampler` は、語彙の頻度が対数一様分布（ジップの法則）に従うことをすでに前提としています。これらの分布重み付きサンプリングを使用すると、ネガティブサンプリングの目的を訓練するためのより単純な損失関数を使用して、Noise Contrastive Estimation（NCE）損失を近似するのにも役立ちます。

### 訓練データの生成

上記で説明したすべてのステップを、任意のテキストデータセットから取得したベクトル化された文のリストに対して呼び出すことができる関数にまとめます。スキップグラムの単語ペアをサンプリングする前にサンプリングテーブルが構築されることに注目してください。この関数は以降のセクションで使用します。

In [ ]:
"""
Generates skip-gram pairs with negative sampling for a list of sequences
(int-encoded sentences) based on window size, number of negative samples
and vocabulary size.
"""


def generate_training_data(sequences, window_size, num_ns, vocab_size, seed):
    # 各訓練サンプルの要素がこれらのリストに追加されます。
    targets, contexts, labels = [], [], []

    # `vocab_size` 個のトークンに対するサンプリングテーブルを構築
    # TODO 1a
    sampling_table = tf.keras.preprocessing.sequence.make_sampling_table(
        vocab_size
    )

    # データセット内のすべてのシーケンス（文）を反復処理
    # TODO 1a: Your code here
    for sequence in tqdm.tqdm(sequences):

        # シーケンス（文）に対して正のスキップグラムペアを生成
        positive_skip_grams, _ = tf.keras.preprocessing.sequence.skipgrams(
            sequence,
            vocabulary_size=vocab_size,
            sampling_table=sampling_table,
            window_size=window_size,
            negative_samples=0,
            seed=seed
        )

        # 各正のスキップグラムペアを反復処理して、
        # 正のコンテキスト単語とネガティブサンプルを含む訓練サンプルを生成
        # TODO 1b
        for target_word, context_word in positive_skip_grams:
            # TODO 1b: your code here
            context_class = ops.expand_dims(
                ops.convert_to_tensor([context_word], dtype="int64"), 1
            )
            negative_sampling_candidates, _, _ = (
                tf.random.log_uniform_candidate_sampler(
                    true_classes=context_class,
                    num_true=1,
                    num_sampled=num_ns,
                    unique=True,
                    range_max=vocab_size,
                    seed=seed,
                    name="negative_sampling",
                )
            )

            # コンテキストとラベルのベクトルを構築（1 つのターゲット単語用）
            context = ops.concatenate(
                [
                    ops.squeeze(context_class, axis=1),
                    negative_sampling_candidates,
                ],
                0,
            )
            label = ops.convert_to_tensor([1] + [0] * num_ns, dtype="int64")

            # 訓練サンプルの各要素をグローバルリストに追加
            targets.append(target_word)
            contexts.append(context)
            labels.append(label)

    return targets, contexts, labels

## ラボ課題 2: Word2Vec 用の訓練データの準備

スキップグラムネガティブサンプリングに基づく Word2Vec モデルで 1 つの文を処理する方法を理解したところで、より大きな文のリストから訓練サンプルの生成に進むことができます。

### テキストコーパスのダウンロード

このチュートリアルではシェイクスピアの著作のテキストファイルを使用します。独自のデータでこのコードを実行するには、次の行を変更してください。

In [ ]:
path_to_file = keras.utils.get_file(
    "shakespeare.txt",
    "https://storage.googleapis.com/download.tensorflow.org/data/shakespeare.txt",
)

ファイルからテキストを読み込み、最初の数行を確認します。

In [ ]:
with open(path_to_file) as f:
    lines = f.read().splitlines()
for line in lines[:20]:
    print(line)

空でない行を使用して、次のステップのための `tf.data.TextLineDataset` オブジェクトを構築します。

In [ ]:
# TODO 2a
text_ds = tf.data.TextLineDataset(path_to_file).filter(
    lambda x: tf.cast(tf.strings.length(x), bool)
)

### コーパスからの文のベクトル化

`TextVectorization` レイヤを使用してコーパスから文をベクトル化できます。このレイヤの使用方法の詳細については、こちらの[テキスト分類](https://www.tensorflow.org/tutorials/keras/text_classification)チュートリアルを参照してください。上記の最初の数文からわかるように、テキストは大文字小文字を統一し、句読点を削除する必要があります。これを行うには、TextVectorization レイヤで使用できる `custom_standardization` 関数を定義します。

In [ ]:
"""
We create a custom standardization function to lowercase the text and
remove punctuation.
"""


def custom_standardization(input_data):
    lowercase = tf.strings.lower(input_data)
    return tf.strings.regex_replace(
        lowercase, "[%s]" % re.escape(string.punctuation), ""
    )


"""
Define the vocabulary size and number of words in a sequence.
"""
vocab_size = 4096
sequence_length = 10

"""
Use the text vectorization layer to normalize, split, and map strings to
integers. Set output_sequence_length length to pad all samples to same length.
"""
vectorize_layer = layers.TextVectorization(
    standardize=custom_standardization,
    max_tokens=vocab_size,
    output_mode="int",
    output_sequence_length=sequence_length,
)

テキストデータセットに対して `adapt` を呼び出して語彙を作成します。

In [ ]:
vectorize_layer.adapt(text_ds.batch(1024))

レイヤの状態がテキストコーパスを表すように適合（adapt）されたら、`get_vocabulary()` を使用して語彙にアクセスできます。この関数は、頻度の高い順（降順）にソートされたすべての語彙トークンのリストを返します。

In [ ]:
# 参照用に作成した語彙を保存
inverse_vocab = vectorize_layer.get_vocabulary()
print(inverse_vocab[:20])

これで、vectorize_layer を使用して `text_ds` 内の各要素のベクトルを生成できるようになりました。

In [ ]:
# text_ds のデータをベクトル化
text_vector_ds = (
    text_ds.batch(1024).prefetch(AUTOTUNE).map(vectorize_layer).unbatch()
)

### データセットからのシーケンスの取得

これで、整数エンコードされた文の `tf.data.Dataset` が得られました。Word2Vec モデルを訓練するためのデータセットを準備するには、データセットを文ベクトルのシーケンスのリストにフラット化します。データセット内の各文を反復処理して正と負のサンプルを生成するため、このステップが必要です。

注: 先ほど定義した `generate_training_data()` は TF 以外の Python / NumPy 関数を使用しているため、`tf.data.Dataset.map()` で `tf.py_function` または `tf.numpy_function` を使用することもできます。

In [ ]:
sequences = list(text_vector_ds.as_numpy_iterator())
print(len(sequences))

`sequences` の例をいくつか見てみましょう。

In [ ]:
for seq in sequences[:5]:
    print(f"{seq} => {[inverse_vocab[i] for i in seq]}")

### シーケンスからの訓練サンプルの生成

`sequences` は整数エンコードされた文のリストになりました。先ほど定義した `generate_training_data()` 関数を呼び出して、Word2Vec モデル用の訓練サンプルを生成するだけです。おさらいすると、この関数は各シーケンスの各単語を反復処理して、正および負のコンテキスト単語を収集します。target、contexts、labels の長さは同じになり、訓練サンプルの総数を表します。

In [ ]:
targets, contexts, labels = generate_training_data(
    sequences=sequences,
    window_size=2,
    num_ns=4,
    vocab_size=vocab_size,
    seed=SEED,
)

targets = np.array(targets)
contexts = np.array(contexts)
labels = np.array(labels)

print("\n")
print(f"targets.shape: {targets.shape}")
print(f"contexts.shape: {contexts.shape}")
print(f"labels.shape: {labels.shape}")

### パフォーマンス向上のためのデータセットの構成

大量の訓練サンプルに対して効率的なバッチ処理を実行するには、`tf.data.Dataset` API を使用します。このステップの後、Word2Vec モデルを訓練するための `(target_word, context_word), (label)` 要素の `tf.data.Dataset` オブジェクトが作成されます。

In [ ]:
BATCH_SIZE = 1024
BUFFER_SIZE = 10000
dataset = tf.data.Dataset.from_tensor_slices(((targets, contexts), labels))
dataset = dataset.shuffle(BUFFER_SIZE).batch(BATCH_SIZE, drop_remainder=True)
print(dataset)

パフォーマンスを向上させるために `cache()` と `prefetch()` を追加します。

In [ ]:
dataset = dataset.cache().prefetch(buffer_size=AUTOTUNE)
print(dataset)

## ラボ課題 3: モデルと訓練

Word2Vec モデルは、スキップグラムからの真のコンテキスト単語と、ネガティブサンプリングによって得られた偽のコンテキスト単語を区別する分類器として実装できます。ターゲット単語とコンテキスト単語の埋め込みの間で内積を計算してラベルの予測を取得し、データセット内の真のラベルに対する損失を計算できます。

### サブクラス化された Word2Vec モデル

[Keras Subclassing API](https://www.tensorflow.org/guide/keras/custom_layers_and_models) を使用して、次のレイヤを持つ Word2Vec モデルを定義します。


* `target_embedding`: 単語がターゲット単語として現れたときにその単語の埋め込みを検索する `keras.layers.Embedding` レイヤ。このレイヤのパラメータ数は `(vocab_size * embedding_dim)` です。
* `context_embedding`: 単語がコンテキスト単語として現れたときにその単語の埋め込みを検索する別の `keras.layers.Embedding` レイヤ。このレイヤのパラメータ数は `target_embedding` と同じ、つまり `(vocab_size * embedding_dim)` です。
* `dots`: 訓練ペアからのターゲット埋め込みとコンテキスト埋め込みの内積を計算する `ops.einsum` レイヤ。

サブクラス化されたモデルを使用すると、`(target, context)` ペアを受け取り、対応する埋め込みレイヤに渡す `call()` 関数を定義できます。`context_embedding` をリシェイプして `target_embedding` との内積を計算し、フラット化された結果を返します。

重要ポイント: `target_embedding` レイヤと `context_embedding` レイヤは共有することもできます。また、両方の埋め込みを連結したものを最終的な Word2Vec 埋め込みとして使用することもできます。

In [ ]:
class Word2Vec(keras.Model):
    def __init__(self, vocab_size, embedding_dim):
        super().__init__()
        self.target_embedding = layers.Embedding(
            vocab_size, embedding_dim, name="w2v_embedding"
        )
        self.context_embedding = layers.Embedding(vocab_size, embedding_dim)

    def call(self, pair):
        target, context = pair
        # tf.squeeze の代わりに ops.squeeze を使用
        if len(target.shape) == 2:
            target = ops.squeeze(target, axis=1)

        word_emb = self.target_embedding(target)
        context_emb = self.context_embedding(context)

        # バックエンドに依存しない内積のために ops.einsum を使用
        # (batch, embed), (batch, context, embed) -> (batch, context)
        dots = ops.einsum("be,bce->bc", word_emb, context_emb)
        return dots

### 損失関数の定義とモデルのコンパイル

簡単にするため、ネガティブサンプリング損失の代替として `keras.losses.CategoricalCrossentropy` を使用できます。独自のカスタム損失関数を作成したい場合は、次のように記述することもできます。

``` python
def custom_loss(x_logit, y_true):
      return ops.binary_crossentropy(labels, logits=x_logit, from_logits=y_true)
```

モデルを構築しましょう。埋め込み次元を 128 として Word2Vec クラスをインスタンス化します（異なる値で実験することもできます）。`keras.optimizers.Adam` オプティマイザでモデルをコンパイルします。

In [ ]:
embedding_dim = 128
word2vec = Word2Vec(vocab_size, embedding_dim)

word2vec.compile(
    optimizer=keras.optimizers.Adam(),
    loss=keras.losses.CategoricalCrossentropy(from_logits=True),
    metrics=["accuracy"],
)

また、TensorBoard 用に訓練統計をログ記録するコールバックを定義します。

In [ ]:
tensorboard_callback = tf.keras.callbacks.TensorBoard(log_dir="logs")

上記で準備した `dataset` を使用して、いくつかのエポックの間モデルを訓練します。

In [ ]:
dataset

In [ ]:
word2vec.fit(dataset, epochs=50, callbacks=[tensorboard_callback])

### TensorBoard での訓練の可視化

モデルがどのように訓練されたかを可視化するために、TensorBoard を使用して Word2Vec モデルの精度と損失を表示できます。これを行うには、まずログをローカルから GCS（Cloud Storage）フォルダにコピーする必要があります。

In [ ]:
def copy_tensorboard_logs(local_path: str, gcs_path: str):
    """Copies Tensorboard logs from a local dir to a GCS location.
    After training, batch copy Tensorboard logs locally to a GCS location.
    Args:
      local_path: local filesystem directory uri.
      gcs_path: cloud filesystem directory uri.
    Returns:
      None.
    """
    pattern = f"{local_path}/*/events.out.tfevents.*"
    local_files = tf.io.gfile.glob(pattern)
    gcs_log_files = [
        local_file.replace(local_path, gcs_path) for local_file in local_files
    ]
    for local_file, gcs_file in zip(local_files, gcs_log_files):
        tf.io.gfile.copy(local_file, gcs_file)


copy_tensorboard_logs("./logs", OUTDIR + "/word2vec_logs")

埋め込みを可視化するには、Cloud Shell を開き、次のコマンドを使用します。

In [ ]:
print(f"tensorboard --port=8081 --logdir {OUTDIR}/word2vec_logs")

Cloud Shell で、[ウェブでプレビュー] > [ポートの変更] をクリックし、ポート番号 8081 を入力します。[変更してプレビュー] をクリックして TensorBoard を開きます。

![embeddings_classifier_accuracy.png](assets/embeddings_classifier_accuracy.png)

## ラボ課題 4: 埋め込みの検索と分析

`get_layer()` と `get_weights()` を使用してモデルから重みを取得します。`get_vocabulary()` 関数は、1 行に 1 つのトークンを含むメタデータファイルを構築するための語彙を提供します。

In [ ]:
# TODO 4a
weights = word2vec.get_layer("w2v_embedding").get_weights()[0]
vocab = vectorize_layer.get_vocabulary()

ベクトルとメタデータファイルを保存するための "word2vec" という名前のフォルダを作成します。

In [ ]:
!mkdir word2vec

ベクトルとメタデータファイルを作成して保存します。

In [ ]:
out_v = open("word2vec/vectors.tsv", "w", encoding="utf-8")
out_m = open("word2vec/metadata.tsv", "w", encoding="utf-8")

for index, word in enumerate(vocab):
    if index == 0:
        continue  # 0 はパディングのためスキップ
    vec = weights[index]
    out_v.write("\t".join([str(x) for x in vec]) + "\n")
    out_m.write(word + "\n")
out_v.close()
out_m.close()

`vectors.tsv` と `metadata.tsv` をローカルマシンにダウンロードし、[Embedding Projector](https://projector.tensorflow.org/) を開きます。ここでは、ダウンロードした 2 つのファイルをアップロードして、埋め込みを可視化するオプションが用意されています。

Copyright 2025 Google LLC

Licensed under the Apache License, Version 2.0 (the "License");
you may not use this file except in compliance with the License.
You may obtain a copy of the License at

     https://www.apache.org/licenses/LICENSE-2.0

Unless required by applicable law or agreed to in writing, software
distributed under the License is distributed on an "AS IS" BASIS,
WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
See the License for the specific language governing permissions and
limitations under the License.